In [1]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import RobustScaler
from imblearn.over_sampling import SMOTE
from sklearn.feature_selection import VarianceThreshold, RFECV

RANDOM_STATE = 42
SEED = 42
np.random.seed(SEED)

In [2]:
df = pd.read_excel("../data/preprocessed/telco_preprocessed.xlsx")
df.head()

,b_gender,b_senior,b_partner,b_dependents,i_tenure,b_phone,b_mphone,c_internet,b_onlinesec,b_onlineback,b_devprot,b_techsup,b_streamingtv,b_streamingmovies,c_contract,b_eticket,c_paymethod,f_monthly,f_total,b_churn
0,0,0,1,0,1,0,0,dsl,0,1,0,0,0,0,monthly,1,echeck,29.85,29.85,0
1,1,0,0,0,34,1,0,dsl,1,0,1,0,0,0,1y,0,mcheck,56.95,1889.50,0
2,1,0,0,0,2,1,0,dsl,1,1,0,0,0,0,monthly,1,mcheck,53.85,108.15,1
3,1,0,0,0,45,0,0,dsl,1,0,1,1,0,0,1y,0,btransfer,42.30,1840.75,0
4,0,0,0,0,2,1,0,fiber,0,0,0,0,0,0,monthly,1,echeck,70.70,151.65,1


In [3]:
TARGET = "b_churn"
CAT_COLS = [col for col in df.columns if col.startswith('c_') and col != TARGET]
NUM_COLS = [col for col in df.columns if col.startswith('f_') or col.startswith('i_') and col != TARGET]
BIN_COLS = [col for col in df.columns if col.startswith('b_') and col != TARGET]
STR_COLS = [col for col in df.columns if col.startswith('s_') and col != TARGET]

## 2.2. Imputacion de valores

Vemos que unicamente tenemos valores faltantes en una columna.

In [4]:
df_with_na = df[df.isna().any(axis=1)]
df_with_na

,b_gender,b_senior,b_partner,b_dependents,i_tenure,b_phone,b_mphone,c_internet,b_onlinesec,b_onlineback,b_devprot,b_techsup,b_streamingtv,b_streamingmovies,c_contract,b_eticket,c_paymethod,f_monthly,f_total,b_churn
488,0,0,1,1,0,0,0,dsl,1,0,1,1,1,0,2y,1,btransfer,52.55,NaN,0
753,1,0,0,1,0,1,0,no,0,0,0,0,0,0,2y,0,mcheck,20.25,NaN,0
936,0,0,1,1,0,1,0,dsl,1,1,1,0,1,1,2y,0,mcheck,80.85,NaN,0
1082,1,0,1,1,0,1,1,no,0,0,0,0,0,0,2y,0,mcheck,25.75,NaN,0
1340,0,0,1,1,0,0,0,dsl,1,1,1,1,1,0,2y,0,ccard,56.05,NaN,0
3331,1,0,1,1,0,1,0,no,0,0,0,0,0,0,2y,0,mcheck,19.85,NaN,0
3826,1,0,1,1,0,1,1,no,0,0,0,0,0,0,2y,0,mcheck,25.35,NaN,0
4380,0,0,1,1,0,1,0,no,0,0,0,0,0,0,2y,0,mcheck,20.00,NaN,0
5218,1,0,1,1,0,1,0,no,0,0,0,0,0,0,1y,1,mcheck,19.70,NaN,0
6670,0,0,1,1,0,1,1,dsl,0,1,1,1,1,0,2y,0,mcheck,73.35,NaN,0


Normalmente una buena practica en ciencia de datos en los casos de valores faltantes en una variable numerica, es imputar la media o la mediana en funcion de si la distribucion es simetrica o asimetrica.

In [5]:
def decidir_simetria(
    serie: pd.Series,
    umbral_skew: float = 0.75,
    tol_media_mediana: float = 0.05
) -> dict:
    """
    Decide si una columna numérica es simétrica aplicando:
      1) |skewness| < umbral_skew
      2) diferencia relativa |media - mediana| / (|media| + 1e-9) < tol_media_mediana

    Parámetros
    ----------
    serie : pd.Series
        Columna numérica a evaluar (se ignoran NaN).
    umbral_skew : float, por defecto 0.5
        Umbral absoluto de skewness para considerar simetría.
    tol_media_mediana : float, por defecto 0.05 (5%)
        Tolerancia de proximidad relativa media–mediana.

    Retorna
    -------
    dict con:
        - simetrica : bool
        - skewness  : float
        - media     : float
        - mediana   : float
        - sesgo     : str  ("positivo", "negativo" o "nulo")
        - crit_skew : bool (cumple criterio de skewness)
        - crit_mm   : bool (cumple criterio de media≈mediana)
        - diff_rel  : float (diferencia relativa usada)
        - n         : int   (nº de valores no nulos usados)
    """
    s = serie.dropna()
    n = int(s.shape[0])

    if n == 0:
        return {
            "simetrica": False,
            "skewness": np.nan,
            "media": np.nan,
            "mediana": np.nan,
            "sesgo": "nulo",
            "crit_skew": False,
            "crit_mm": False,
            "diff_rel": np.nan,
            "n": 0
        }

    skew_value = s.skew()          # sesgo muestral
    media = s.mean()
    mediana = s.median()

    # Tipo de sesgo según media vs mediana
    if media > mediana:
        sesgo = "positivo"
    elif media < mediana:
        sesgo = "negativo"
    else:
        sesgo = "nulo"

    # Criterios
    crit_skew = abs(skew_value) < umbral_skew
    diff_rel = abs(media - mediana) / (abs(media) + 1e-9)
    crit_mm = diff_rel < tol_media_mediana

    simetrica = crit_skew and crit_mm

    return {
        "simetrica": simetrica,
        "skewness": float(skew_value),
        "media": float(media),
        "mediana": float(mediana),
        "sesgo": sesgo,
        "crit_skew": bool(crit_skew),
        "crit_mm": bool(crit_mm),
        "diff_rel": float(diff_rel),
        "n": n
    }, simetrica

In [6]:
def imputar_media_o_mediana(df, columna):
    """
    Imputa la media o mediana según la distribución de la columna.
    
    Parámetros:
        df (pd.DataFrame): DataFrame con los datos.
        columna (str): Nombre de la columna float a imputar.
        umbral_sesgo (float): Valor absoluto de sesgo a partir del cual se considera sesgada.
        
    Retorna:
        pd.DataFrame: DataFrame con la columna imputada.
    """
    
    # Calcular sesgo (skewness)
    stats, simetrica =decidir_simetria(df[columna])

    if simetrica:
        valor_imputacion = df[columna].mean()
        metodo = "media"
    else:
        valor_imputacion = df[columna].median()
        metodo = "mediana"
    
    print(f"Columna: {columna} | Método usado: {metodo} | Stats: {stats:.3f}")
    
    df[columna] = df[columna].fillna(valor_imputacion)
    
    return df

In [7]:
_, sim = decidir_simetria(df['f_total'])
print(sim)

False


Vemos que la columna no es simetrica, por lo que deberiamos imputar la mediana. SIn embargo, si prestamos atencion a la logica de negocio, vemos que faltan valores solo cuando la tenure (cantidad de meses desde que la persona se convirtio en cliente) es 0. Por lo tanto, la cantidad todal de dinero pagado hasta la fecha es 0, y este sera el valor que imputaremos.

In [8]:
df['f_total'] = df['f_total'].fillna(0, inplace=False)

## 2.3. Codificacion variables categoricas

Aquellas variables categoricas con mas de dos niveles las codificaremos mediante la tecnica de OneHotEncoding. Como esta tecnica no da lugar a data leakeage, podemos realizarla sobre el conjunto original.

In [9]:
df = pd.get_dummies(df, columns=CAT_COLS, drop_first=False, dtype=int) 
df.head()

,b_gender,b_senior,b_partner,b_dependents,i_tenure,b_phone,b_mphone,b_onlinesec,b_onlineback,b_devprot,...,c_internet_dsl,c_internet_fiber,c_internet_no,c_contract_1y,c_contract_2y,c_contract_monthly,c_paymethod_btransfer,c_paymethod_ccard,c_paymethod_echeck,c_paymethod_mcheck
0,0,0,1,0,1,0,0,0,1,0,...,1,0,0,0,0,1,0,0,1,0
1,1,0,0,0,34,1,0,1,0,1,...,1,0,0,1,0,0,0,0,0,1
2,1,0,0,0,2,1,0,1,1,0,...,1,0,0,0,0,1,0,0,0,1
3,1,0,0,0,45,0,0,1,0,1,...,1,0,0,1,0,0,1,0,0,0
4,0,0,0,0,2,1,0,0,0,0,...,0,1,0,0,0,1,0,0,1,0


## 2.4. Escalado de variables numericas

Antes de escalar, dividiremos el dataset. Como es un problema de clasificacion desbalanceado, ajustamos el hiperparametro stratify con tal de que el conjunto de entrenamiento mantenga este desbalanceo.

In [10]:
X = df.drop(columns=[TARGET])
y = df[TARGET].astype(int)

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,           
    stratify=y, # estratificación para mantener la prevalencia
    random_state=SEED
)

def prevalencia(y):
    return {"prevalencia(%)": round(100*y.mean(), 2)}

print("Distribución de clases")
print("TRAIN:", prevalencia(y_train))
print("TEST :", prevalencia(y_test))

Distribución de clases
TRAIN: {'prevalencia(%)': np.float64(26.54)}
TEST : {'prevalencia(%)': np.float64(26.54)}


Normalmente en el caso de las variables numericas, se utiliza el StandardScaler o MinMAxScaler, pero estos asumen que la distribucion de la variable es normal. Sin embargo, en la seccion 1.6 del notebook 01_eda hemos visto que las tres parecian ser asimetricas, vamos a comprobarlo.

In [11]:
for col in NUM_COLS:
    _, sim = decidir_simetria(df[col])
    print(f'La columna {col} es simetrica? {sim}')

La columna i_tenure es simetrica? False
La columna f_monthly es simetrica? False
La columna f_total es simetrica? False


Por lo tanto, vamos a utilizar un Scaler mas apropiado para distribuciones asimetricas.

In [12]:
scaler = RobustScaler()
X_train_scaled = X_train.copy()
X_train_scaled[NUM_COLS] = scaler.fit_transform(X_train_scaled[NUM_COLS])
X_test_scaled = X_test.copy()
X_test_scaled[NUM_COLS] = scaler.transform(X_test_scaled[NUM_COLS])

## 2.5. Preparacion final de los datos

Vamos a explorar distintos enfoques que cambian sustancialmente la estrucutra de los datos:

- En primer lugar, dado que se trata de un conjunto de datos desbalanceado, exploraremos la opcion de generar datos sinteticamente mediante la tecnica SMOTE.
- Al conjunto resultante, aplicaremos tecnicas de feature selection.


### 2.5.1. SMOTE

In [13]:
print("Distribución antes de SMOTE:", np.bincount(y_train))
smote = SMOTE(random_state=42, k_neighbors=5)
X_train_bal, y_train_bal = smote.fit_resample(X_train_scaled, y_train)
print("Distribución después de SMOTE:", np.bincount(y_train_bal))

Distribución antes de SMOTE: [4139 1495]
Distribución después de SMOTE: [4139 4139]


### 2.5.2. Seleccion de caracteristicas

#### i. Varianza

In [14]:
def sel_varianza(X_train):
    """
    Elimina características con baja varianza del conjunto de entrenamiento.

    **Procedimiento:**
    - Aplica un umbral de varianza para identificar y eliminar características con varianza inferior a 0.001.
    - Crea un nuevo DataFrame con las características seleccionadas.
    - Identifica y muestra las columnas que no pasaron el umbral de varianza.

    **Finalidad:**
    Reducir el número de características eliminando aquellas que no aportan información significativa debido a su baja variabilidad.
    """
    # Eliminar características con baja varianza
    selector = VarianceThreshold(threshold=0.001)
    X_var = selector.fit_transform(X_train)
    X_var = pd.DataFrame(X_var, columns=X_train.columns[selector.get_support()])

    # Obtener las columnas que no pasaron el umbral de varianza (False)
    low_variance_columns = X_train.columns[~selector.get_support()]

    # Mostrar los nombres de las columnas con baja varianza
    if len(low_variance_columns) > 0:
        print("Columnas con baja varianza (se eliminarán):")
        print(low_variance_columns.tolist())
    else:
        print("No hay columnas con baja varianza.")

    return low_variance_columns

#### iii. Corrrelacion

In [15]:
def sel_correlacion(X_sel, y_sel, thres: float):
    """
    Selecciona características por correlación absoluta con la variable objetivo.
    Devuelve:
      - corrs: DataFrame con columna 'vals' (|corr|) ordenado desc.
      - selected_features: lista de columnas con |corr| > thres
    """
    # Asegura y como Serie 1D (independiente del nombre de columna)
    if isinstance(y_sel, pd.DataFrame):
        y_vec = y_sel.iloc[:, 0]
    else:
        y_vec = y_sel

    # Correlación de cada feature con el target
    correlations = X_sel.apply(lambda x: x.corr(y_vec))

    # DataFrame ordenado por |corr|
    corrs = pd.DataFrame({'vals': correlations.abs().values}, index=correlations.index)\
             .sort_values('vals', ascending=False)

    # Selección SIN mezclar índices externos
    selected_features = corrs.index[corrs['vals'] > thres].tolist()

    return corrs, selected_features


#### iii. Colinealidad

In [16]:
def sel_colinealidad(X_train, thres, y_train, v_corr):
    """
    Identifica pares de características con alta colinealidad y sugiere cuál eliminar basándose en la correlación con la variable objetivo.

    **Procedimiento:**
    - Calcula la matriz de correlación absoluta entre las características.
    - Crea la matriz triangular superior para evitar pares duplicados.
    - Identifica pares de características con correlación superior al umbral especificado.
    - Para cada par, compara la correlación de cada característica con la variable objetivo.
    - Sugiere eliminar la característica con menor correlación absoluta con la variable objetivo.
    - Imprime información sobre los pares y las características sugeridas para eliminar.

    **Finalidad:**
    Reducir la multicolinealidad en el conjunto de datos, mejorando la interpretabilidad y rendimiento del modelo.
    """
    # Paso 1: Calcular la matriz de correlación absoluta
    corr_matrix = X_train.corr().abs()

    # Paso 2: Crear la matriz triangular superior para evitar pares duplicados
    upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))

    # Paso 3: Identificar pares de características con alta correlación
    high_corr_pairs = [(row, column) for row in upper.index for column in upper.columns if upper.loc[row, column] > thres]

    features_to_drop = set()

    corrs, cols_high_corr = sel_correlacion(X_train, y_train, v_corr)
    target_correlations = corrs['vals']
    
    # Paso 4: Para cada par, decidir qué característica eliminar
    for feature1, feature2 in high_corr_pairs:
        corr1 = target_correlations.get(feature1, 0)
        corr2 = target_correlations.get(feature2, 0)

        # Decidir qué característica eliminar
        if abs(corr1) > abs(corr2):
            feature_to_drop = feature2
        else:
            feature_to_drop = feature1

        features_to_drop.add(feature_to_drop)

        print(f"Pares altamente correlacionados: {feature1} y {feature2} (corr: {corr_matrix.loc[feature1, feature2]:.2f})")
        print(f"Correlación con objetivo - {feature1}: {corr1:.2f}, {feature2}: {corr2:.2f}")
        print(f"Sugerencia de eliminar: {feature_to_drop}\n")

    return list(features_to_drop)


#### iv. Statistical Selector

In [17]:
def statistical_select(X_train_transformed, y_train, X_test_transformed, v_corr, v_col):
    """
    Realiza la selección de características estadística basada en varianza, correlación y colinealidad.

    **Finalidad:**
    Reducir el número de características eliminando aquellas con baja varianza, baja correlación con la variable objetivo y alta colinealidad, mejorando así la eficiencia y precisión del modelo.

    **Procedimiento:**
    - Elimina las características con baja varianza utilizando `sel_varianza`.
    - Calcula la correlación de las características con la variable objetivo y selecciona las que superan un umbral utilizando `sel_correlacion`.
    - Identifica y elimina pares de características altamente correlacionadas utilizando `sel_colinealidad`.
    - Excluye manualmente algunas características específicas.
    - Devuelve el DataFrame filtrado con las características seleccionadas.
    """
    cols_low_var = sel_varianza(X_train_transformed)

    corrs, cols_high_corr = sel_correlacion(X_train_transformed, y_train, v_corr)
    cols = list(set(cols_high_corr) - set(cols_low_var))

    cols_high_col = sel_colinealidad(X_train_transformed[cols], v_col, y_train, v_corr)
    cols = list(set(cols) - set(cols_high_col))

    X_train_selected = X_train_transformed[cols]
    X_test_selected = X_test_transformed[cols]

    print("Statistical - Numero de caracteristicas seleccionadas: ", X_train_selected.shape[1], ", inicialmente ", X_train_transformed.shape[1])

    return X_train_selected, X_test_selected

#### v. Random Forest Selector

In [18]:
def rfr_select(X_train_scaled, y_train, X_test_scaled):
    """
    Realiza la selección de características utilizando RFECV con RandomForestRegressor.

    **Finalidad:**
    Seleccionar las características más relevantes del conjunto de datos utilizando la Eliminación Recursiva de Características con Validación Cruzada (RFECV) y un modelo de Random Forest Regressor.

    **Procedimiento:**
    - Estandariza las variables de entrenamiento y prueba utilizando `StandardScaler`.
    - Define un estimador de Random Forest para ser utilizado en RFECV.
    - Configura RFECV especificando el estimador, el número de pasos, validación cruzada, métrica de puntuación y número mínimo de características a seleccionar.
    - Ajusta RFECV en el conjunto de entrenamiento para determinar las características óptimas.
    - Obtiene los índices y nombres de las características seleccionadas.
    - Crea nuevos DataFrames de entrenamiento y prueba que incluyen solo las características seleccionadas.
    """

    rf_estimator = RandomForestClassifier(random_state=42, n_estimators=200)

    rfecv = RFECV(
        estimator=rf_estimator,
        step=1,
        cv=5,
        scoring='roc_auc',          # <- era 'neg_mean_squared_error' (regresión)
        min_features_to_select=1,
        n_jobs=-1
    )

    # y como 1D para sklearn
    y_1d = np.ravel(y_train)  # también vale: y_train.squeeze()

    rfecv.fit(X_train_scaled, y_1d)

    idx = np.where(rfecv.support_)[0]
    names = X_train_scaled.columns[idx]

    X_train_selected = pd.DataFrame(X_train_scaled.iloc[:, idx], columns=names)
    X_test_selected  = pd.DataFrame(X_test_scaled.iloc[:,  idx], columns=names)

    print(f"Forest - Número de características seleccionadas: {X_train_selected.shape[1]}, "
          f"originalmente {X_train_scaled.shape[1]}")

    return X_train_selected, X_test_selected

### 2.5.3. Datasets resultantes

In [19]:
X_train_bal = pd.DataFrame(X_train_bal, columns=X_train.columns)
y_train_bal = pd.DataFrame(y_train_bal, columns=[TARGET])

X_train_scaled = pd.DataFrame(X_train_scaled, columns=X_train.columns)
X_test_scaled = pd.DataFrame(X_test_scaled, columns=X_test.columns)

y_train = pd.DataFrame(y_train)
y_test = pd.DataFrame(y_test)

#### i. Con SMOTE y con Feature Selection

In [20]:
X_train_selected1, X_test_selected1 = statistical_select(X_train_bal, y_train_bal, X_test_scaled,
                                                         v_corr=0.01, v_col=0.95)

X_train_selected2, X_test_selected2 = rfr_select(X_train_selected1, y_train_bal, X_test_selected1)

X_train_selected2.to_excel("../data/processed/X_train_cc.xlsx", index=False)
X_test_selected2.to_excel("../data/processed/X_test_cc.xlsx", index=False)

y_train_bal.to_excel("../data/processed/y_train_cc.xlsx", index=False)
y_test.to_excel("../data/processed/y_test_cc.xlsx", index=False)

No hay columnas con baja varianza.


Statistical - Numero de caracteristicas seleccionadas:  26 , inicialmente  26
Forest - Número de características seleccionadas: 26, originalmente 26


#### ii. Con SMOTE y sin Feature Selection

In [21]:
X_train_bal.to_excel("../data/processed/X_train_cs.xlsx", index=False)
X_test_scaled.to_excel("../data/processed/X_test_cs.xlsx", index=False)

y_train_bal.to_excel("../data/processed/y_train_cs.xlsx", index=False)
y_test.to_excel("../data/processed/y_test_cs.xlsx", index=False)

#### iii. Sin SMOTE y con Feature Selection

In [22]:
X_train_selected1, X_test_selected1 = statistical_select(X_train_scaled, y_train, X_test_scaled,
                                                         v_corr=0.01, v_col=0.95)

X_train_selected2, X_test_selected2 = rfr_select(X_train_selected1, y_train, X_test_selected1)

X_train_selected2.to_excel("../data/processed/X_train_sc.xlsx", index=False)
X_test_selected2.to_excel("../data/processed/X_test_sc.xlsx", index=False)

y_train.to_excel("../data/processed/y_train_sc.xlsx", index=False)
y_test.to_excel("../data/processed/y_test_sc.xlsx", index=False)

No hay columnas con baja varianza.
Statistical - Numero de caracteristicas seleccionadas:  25 , inicialmente  26
Forest - Número de características seleccionadas: 23, originalmente 25


#### iv. Sin SMOTE y sin Feature Selection

In [23]:
X_train_scaled.to_excel("../data/processed/X_train_ss.xlsx", index=False)
X_test_scaled.to_excel("../data/processed/X_test_ss.xlsx", index=False)

y_train.to_excel("../data/processed/y_train_ss.xlsx", index=False)
y_test.to_excel("../data/processed/y_test_ss.xlsx", index=False)

In [25]:
print(X_train_scaled.shape, y_train.shape)

(5634, 26) (5634, 1)
